In [3]:
library(Seurat)
library(nicheDE)
library(qs)
library(glue)
library(ggplot2)
library(patchwork)
library(latex2exp)
library(ggtree)
library(paletteer)
library(scCustomize)
library(scico)
library(readxl)
library(ComplexHeatmap)
library(ggsci)
library(latex2exp)
library(ggrepel)
library(org.Hs.eg.db)
library(clusterProfiler)
library(RColorBrewer)
library(Polychrome)
library(stringr) 
library(parallel)
library(ggpubr)
library(dplyr)
library(tidyr)

In [4]:
colors_all <- c(
  "#DA80DA", "#815481", "#C040C0", "#E1AFE1", "#3F0034",
  "#EDABB9", "#EB5C79", "#A06A75", "#C00028",
  "#EB675E", "#A23E36",
  "#540F54", "#53407F",
  "#DFA38A", "#8C3612", "#623623", "#916350", "#DAC3C3",
  "#F8770B", "#E09E3A", "#CD7225", "#FFC990", "#AC5812",
  "#FEE083", "#897538", "#E7B419", "#BCA048",
  "#6F8BE2", "#3053BC",
  "#6D9F58", "#9EB766", "#BDCB10", "#3A6527", "#9EA743",
  "#E2E8A7", "#5A6209", "#8FE36B",
  "#818A31",
  "#9FC5E8", "#23D9F1",
  "#64C6A6",
  "#AAAAAA"
)
ct_cols <- c("M2"=colors_all[10],
             "M5"=colors_all[28],
             "F2"=colors_all[30],
             "F7"=colors_all[24],
             "M1"=colors_all[4],
             "NK"=colors_all[39],
             "M10"=colors_all[17],
             "M3"=colors_all[32],
             "F3"=colors_all[29],
             "F4"=colors_all[12],
             "B3"=colors_all[41],
             "E0"=colors_all[19],
             "B1"=colors_all[7],
             "B2"=colors_all[11])

In [14]:
spatialVariablePlot <- function(obj, plot_cts, focal_ct=NULL, focal_size=0.85){
    coords <- GetTissueCoordinates(obj)
    rownames(coords) <- coords$cell
    ap_ratio <- diff(range(coords[, "x"]))/diff(range(coords[, "y"]))
    scale_bar_x_start <- min(coords[, "y"]) + 50
    scale_bar_x_end <- min(coords[, "y"]) + 50 + 1000
    scale_bar_y_pos <- min(coords[, "x"] + 50)
    coords |> dplyr::select(-cell) -> coords
    cells_of_interest <- which(Idents(obj) %in% plot_cts)
    coords$celltype <- "Background"
    coords$celltype[cells_of_interest] <- as.character(Idents(obj)[cells_of_interest])
    coords$celltype <- factor(coords$celltype, levels=c("Background", sort(unique(as.character(Idents(obj)[cells_of_interest])))))
    plot_order <- seq_len(nlevels(coords$celltype))
    names(plot_order) <- levels(coords$celltype)
    coords$plot_order <- plot_order[as.character(coords$celltype)]
    coords %>%
    arrange(plot_order) -> coords
    alpha_vals <- rep(1, nlevels(coords$celltype))
    ct_palalpha_vals <- rep(1, nlevels(coords$celltype))
    names(alpha_vals) <- levels(coords$celltype)
    alpha_vals["Background"] <- 0.2
    size_vals <- rep(0.4, nlevels(coords$celltype))
    names(size_vals) <- levels(coords$celltype)
    if(length(focal_ct) > 0){
       size_vals[focal_ct] <- focal_size 
    }
    ggplot(coords, aes(y, x, fill=celltype, alpha=celltype, size=celltype)) +
                    geom_point(stroke=0.05, shape=21, color="black") +
                    scale_fill_manual(values=ct_cols,
                                      breaks=setdiff(levels(coords$celltype), "Background")) +
                    scale_alpha_manual(values=alpha_vals,
                                       breaks=setdiff(levels(coords$celltype), "Background"),
                                       name="") +
                    scale_size_manual(values=size_vals,
                                      breaks=setdiff(levels(coords$celltype), "Background"),
                                      name="") +
                    theme_void() +
                    theme(legend.key.size=unit(0.3, 'cm'),
                          aspect.ratio=ap_ratio,
                          legend.text=element_text(size=4),
                          plot.title = element_text(size=6)) +
                    labs(color="", fill="", size="") +
                    guides(fill = guide_legend(override.aes=list(size=2, alpha=1))) +
                    annotate("segment",
                             x=scale_bar_x_start,
                             xend=scale_bar_x_end,
                             y=scale_bar_y_pos,
                             yend=scale_bar_y_pos,
                             linewidth=0.5) +
                    annotate("text",
                             x=(scale_bar_x_start+scale_bar_x_end)/2,
                             y=scale_bar_y_pos,
                             label=paste0(1, " mm"),
                             vjust=1.5,
                             size=1.5)
}
spatialValuePlot <- function(df,
                             variable,
                             x_column="centroid_x",
                             y_column="centroid_y",
                             pt.size=0.2,
                             stroke=0.05,
                             ct.column="",
                             midpoint=NA,
                             plt.ct="",
                             title=variable,
                             palette=paletteer_c("viridis::inferno", n=30),
                             order=TRUE,
                             legend_title=variable,
                             custom_scale=FALSE,
                             scale_externel=NULL){
    mid_rescaler <- function(mid) {
      function(x, to = c(0, 1), from = range(x, na.rm = TRUE)) {
        scales::rescale_mid(x, to, from, mid)
      }
    }
    rescaler <- if (is.na(midpoint)) scales::rescale else mid_rescaler(midpoint)
    sp_aspect_ratio <- diff(range(df[, y_column]))/diff(range(df[, x_column]))
    if(nchar(plt.ct) > 0 & nchar(ct.column) > 0){
        df <- df %>%
        filter(!!sym(ct.column)==plt.ct)
    }
    if(order){
        df <- df %>%
        arrange(!!sym(variable))
    }
    color_bar <- if(custom_scale){scale_externel}else{scale_fill_gradientn(colors=palette, rescale=rescaler)}
    scale_bar_x_start <- min(df[, x_column, drop=TRUE]) + 50
    scale_bar_x_end <- min(df[, x_column, drop=TRUE]) + 50 + 1000
    scale_bar_y_pos <- min(df[, y_column, drop=TRUE] + 50)
    ggplot(df, aes(!!sym(x_column), !!sym(y_column), fill=!!sym(variable))) +
    geom_point(stroke=stroke, size=pt.size, shape=21) +
    theme_bw() +
    theme(aspect.ratio=sp_aspect_ratio,
          legend.key.size=unit(0.2, "cm"),
          legend.title=element_text(size=8, face="bold"),
          legend.text=element_text(size=5),
          panel.grid = element_blank(),
          axis.text=element_blank(),
          axis.ticks=element_blank(),
          plot.title=element_text(size=8, face="bold")) +
    color_bar +
    labs(x="sp1",
         y="sp2",
         fill=legend_title) +
    ggtitle(title) +
    annotate("segment",
             x=scale_bar_x_start,
             xend=scale_bar_x_end,
             y=scale_bar_y_pos,
             yend=scale_bar_y_pos,
             linewidth=0.5) +
    annotate("text",
             x=(scale_bar_x_start+scale_bar_x_end)/2,
             y=scale_bar_y_pos,
             label=paste0(1, " mm"),
             vjust=1.5,
             size=1.5)
}

In [6]:
all_ra_samples <- list.dirs("/data1/deyk/harry/RA_Xenium/data/post_qc_data_baysor/", recursive=FALSE, full.names=TRUE)
all_sample_ids <- str_extract(all_ra_samples, "RA\\d+[A-Z]*")
height_width_param <- c("6&8", "6.5&7", "6&8", "5&9", "6&8", "7.5&4", "6&10", "6&10", "8&10", "6&8")
names(height_width_param) <- c("RA401", "RA331", "RA442", "RA443", "RA457", "RA480", "RA494", "RA519", "RA362", "RA489")

# 4B cell types

In [7]:
plot_M2_M5_F7_F2 <- lapply(all_sample_ids, function(sample){
    figure_height_width <- as.numeric(str_split(height_width_param[sample], "&")[[1]])
    obj <- qread(glue("/data1/deyk/harry/RA_Xenium/data/post_qc_data_baysor/{sample}/final_version_seurat_obj.qs"))
    plot.tmp <- spatialVariablePlot(obj, plot_cts=c("M2", "M5", "F2", "F7"), focal_ct="F7")
    ggsave(plot=plot.tmp,
           filename=glue("/data1/deyk/harry/RA_Xenium/results/figures/manuscript_final_version_figures/figure4/4B_{sample}_cts.png"),
           height=figure_height_width[1],
           width=figure_height_width[2],
           dpi=500,
           limitsize=FALSE,
           bg="white")
})

# 4B fibrinolysis

In [8]:
kmeans_res <- qread("/data1/deyk/harry/RA_Xenium/results/nicheassay/kmeans_res/k_5_res.qs")
kmeans_res <- kmeans_res$cluster
kmeans_niche_df <- data.frame(Niche=kmeans_res) %>%
    mutate(sample=gsub("_.+", "", names(kmeans_res)))
kmeans_niche_df <- split(kmeans_niche_df, kmeans_niche_df$sample)
kmeans_niche_df <- lapply(kmeans_niche_df, function(df){
    rownames(df) <- gsub("RA[0-9]+_", "", rownames(df))
    df %>% 
    mutate(Niche=factor(Niche))
})
all_xenium_samples <- lapply(all_sample_ids, function(sample){
    qread(glue("/data1/deyk/harry/RA_Xenium/data/post_qc_data_baysor/{sample}/final_version_seurat_obj.qs"))
})
all_xenium_samples <- lapply(all_xenium_samples, function(obj){
    # discard assays we don't need, merging will be faster this way
    DefaultAssay(obj) <- "Xenium"
    obj[["SCT"]] <- NULL
    obj[["ControlCodeword"]] <- NULL
    obj[["ControlProbe"]] <- NULL
    coords <- GetTissueCoordinates(obj) %>%
        tibble::column_to_rownames("cell")
    obj@images <- list()
    obj <- AddMetaData(obj, coords)
    return(obj)
})
all_xenium_samples <- setNames(all_xenium_samples, all_sample_ids)
all_xenium_samples <- lapply(names(all_xenium_samples), function(sample){
    AddMetaData(all_xenium_samples[[sample]], kmeans_niche_df[[sample]])
})
all_xenium_merged <- merge(all_xenium_samples[[1]], all_xenium_samples[-1])
all_xenium_merged[["Xenium"]] <- JoinLayers(all_xenium_merged[["Xenium"]])
all_xenium_merged <- NormalizeData(all_xenium_merged, normalization.method="LogNormalize")

Warning message:
“Some cell names are duplicated across objects provided. Renaming to enforce unique cell names.”
Normalizing layer: counts



In [9]:
fibrolysis_genes <- read_xlsx("/data1/deyk/harry/RA_Xenium/data/Ian_gene_list_xenium_experiments/SAMac_fibrolysis.xlsx")
fibrolysis_genes <- fibrolysis_genes$`Fibrinolysis program`
fibrolysis_genes <- fibrolysis_genes[!is.na(fibrolysis_genes)]
all_xenium_merged <- AddModuleScore(all_xenium_merged, name="Fibrolysis", features=list("Fibrolysis"=fibrolysis_genes), ctrl=20, nbin=24)

In [11]:
all_xenium_merged@meta.data %>%
    filter(celltype_broad=="myeloid") -> all_myeloid_plt

In [15]:
all_fibrolysis_module_score <- lapply(all_sample_ids, function(sample_id){
    figure_height_width <- as.numeric(str_split(height_width_param[sample_id], "&")[[1]])
    spatialValuePlot(all_myeloid_plt %>% filter(sample==sample_id),
                     palette=paletteer_c("pals::coolwarm", n=50),
                     midpoint=0,
                     pt.size=0.4,
                     stroke=0.01,
                     variable="Fibrolysis1",
                     x_column="y",
                     y_column="x",
                     legend_title="Fibrolysis program",
                     title=sample_id) -> module_score.plt
    ggsave(plot=module_score.plt,
           filename=glue("/data1/deyk/harry/RA_Xenium/results/figures/manuscript_final_version_figures/figure4/4B_fibrinolysis_{sample_id}.png"),
           height=figure_height_width[1],
           width=figure_height_width[2],
           dpi=800,
           limitsize=FALSE)
})